In [ ]:
# Imports
import math
import numpy as np
import scipy as sp
from scipy.signal import lti
from scipy.interpolate import interp2d
import matplotlib.pyplot as plt

# Implementation and Testing of Turbulence Models for the F18-HARV Simulation

[Paper][https://ntrs.nasa.gov/api/citations/19980028448/downloads/19980028448.pdf]

The above paper outines an approach used by NASA to model the wind turbulence for the simulation of the F18-HARV. This Notebook serves to duplicate the work to prove validity of the process.



In [ ]:
Tnu = 0.0125
fs = 1 / Tnu # sample rate, number of samples per unit of time

V = 100
bwing = 37.42
Lu = 1750
Lwv = 1750

sigma = 5

Npts = 200

ft = np.logspace(-2, 2, Npts)
w = 2.0 * math.pi * ft
coswt = np.cos(w * Tnu)

tauu = Lu/V
tauwv = Lwv / V






In [ ]:
# Power Spectral Density, Dryden Theory
Ku = sigma**2 * tauu / math.pi
Su = Ku / (1 + (tauu * w)**2)
Su = 10 * np.log10(Su)


plt.semilogx(ft, Su)
plt.grid()
plt.xlabel('frequency (Hz)')
plt.ylabel('PSD (dB)')
plt.xlim([0.01, 100])
plt.ylim([-80, 20])

In [ ]:
# Power Spectral Density, Mil-STD-Dryden

au = Tnu / tauu
onemau = 1 - au

milunum = sigma**2 * Tnu**2 / (math.pi * tauu)

milud = 1 + onemau**2 - 2 * onemau * coswt

Gmilu = 10 * np.log10(milunum / milud)

plt.semilogx(ft, Gmilu, label='Theoretical-MIL-STD')
plt.vlines(fs, -80,20, label='sample frequency')
plt.legend()
plt.grid()
plt.grid()
plt.xlabel('frequency (Hz)')
plt.ylabel('PSD (dB)')
plt.xlim([0.01, 100])
plt.ylim([-80, 20])

In [ ]:
noise_intensity = 0;
sdnoise = 1;

t_s = np.arange(0, 100 + Tnu, Tnu)

noise_u = np.random.normal(noise_intensity, sdnoise, t_s.shape[0])
noise_v = np.random.normal(noise_intensity, sdnoise, t_s.shape[0])
noise_w = np.random.normal(noise_intensity, sdnoise, t_s.shape[0])

plt.plot(t_s, noise_u)

In [ ]:
# Transfer Function Solution
b_u     = sigma * math.sqrt(2 * tauu / math.pi )
b0_u    = b_u
as_u    = tauu
a0_u    = 1

H_u = lti(  [b0_u],
            [as_u, a0_u])

Milu = H_u.output(np.sqrt(math.pi/Tnu) * noise_u,	t_s)[1]

plt.plot(t_s, Milu)

In [ ]:
from scipy.signal.windows import hann

nblock = 1024
overlap = 512
win = hann(nblock, False)

# the scaling factor is already done for us in the scipy welch function so thi calcualtion is unneccessary
SF = np.linalg.norm(win)**2 / np.sum(win)**2

In [ ]:
# PSD of Estimated:
fm, Pmilud = sp.signal.welch(Milu, fs, window=win, noverlap=overlap, nfft=nblock, scaling='spectrum', detrend=False)


Pmilud = 10*np.log10(Pmilud)

plt.semilogx(fm, Pmilud, '.', label='Sample')
plt.semilogx(ft, Gmilu, label='Theoretical Mil Std')
plt.semilogx(ft, Su, label='Theoretical Dryden')
plt.vlines(fs, -80,20, label='sample frequency')
plt.legend()
plt.grid()
plt.xlabel('frequency (Hz)')
plt.ylabel('PSD (dB)')
plt.xlim([0.01, 100])
plt.ylim([-80, 20])

Compare this to the image from the paper:

<img src='./img/f18_psd_u.png' width="40%" height="40%">


In [ ]:
# Power Spectral Density, Dryden Theory
Kwv = sigma**2 * tauwv / (2*math.pi)
Swv = Kwv * (1 + 3 * (tauwv * w)**2) / (1 + (tauwv * w)**2)**2
Swv = 10 * np.log10(Swv)


plt.semilogx(ft, Swv)
plt.grid()
plt.xlabel('frequency (Hz)')
plt.ylabel('PSD (dB)')
plt.xlim([0.01, 100])
plt.ylim([-80, 20])

In [ ]:
# Power Spectral Density, Mil-STD-Dryden

awv = 2.0 * Tnu / tauwv
onemawv = 1 - awv


milwvnum = sigma**2 * Tnu**2 / (math.pi * tauwv)

milwvd = 1 + onemawv**2 - 2 * onemawv * coswt

Gmilwv = 10 * np.log10(milwvnum / milwvd)

plt.semilogx(ft, Gmilwv, label='Theoretical-MIL-STD')
plt.vlines(fs, -80,20, label='sample frequency')
plt.legend()
plt.grid()
plt.xlabel('frequency (Hz)')
plt.ylabel('PSD (dB)')
plt.xlim([0.01, 100])
plt.ylim([-80, 20])

In [ ]:
# Transfer Function Solution
b_v     = sigma * math.sqrt(tauwv / math.pi)
bs_v    = b_v * math.sqrt(3.0) * tauwv
b0_v    = b_v
as2_v   = tauwv ** 2
as_v    = 2 * tauwv
a0_v    = 1

H_v = lti(  [bs_v, b0_v],
            [as2_v, as_v, a0_v])

milvk = H_v.output(np.sqrt(math.pi/Tnu) * noise_v, t_s)[1]

plt.plot(t_s, milvk)

In [ ]:
fm, Pmilvd = sp.signal.welch(milvk, fs, window=win, noverlap=overlap, nfft=nblock, scaling='spectrum', detrend=False)
Pmilvd = 10*np.log10(Pmilvd)

plt.semilogx(fm, Pmilvd, '.', label='Sample')
plt.semilogx(ft, Gmilwv, label='Theoretical-MIL-STD')
plt.semilogx(ft, Swv, label='Theoretical-Dryden')
plt.vlines(fs/2, -80,20, label='50% sample frequency')
plt.legend()
plt.grid()
plt.xlabel('frequency (Hz)')
plt.ylabel('PSD (dB)')
plt.xlim([0.01, 100])
plt.ylim([-80, 20])

Compare to:


<img src='img/f18_psd_v.png' width="40%" height="40%">

In [ ]:
b_w     = sigma * math.sqrt(tauwv / math.pi)
bs_w    = b_w * math.sqrt(3.0) * tauwv
b0_w    = b_w
as2_w   = tauwv ** 2
as_w    = 2 * tauwv
a0_w    = 1

H_w = lti( [bs_w, b0_w], 
           [as2_w, as_w, a0_w])

milwk = H_w.output(np.sqrt(math.pi/Tnu) * noise_w, t_s)[1]

plt.plot(t_s, milwk)

In [ ]:
fm, Pmilwd = sp.signal.welch(milwk, fs, window=win, noverlap=overlap, nfft=nblock, scaling='spectrum', detrend=False)

Pmilwd = 10*np.log10(Pmilwd)
# f_spec, psd_spec_u = sp.signal.welch(Milu, fs, scaling='spectrum')

plt.semilogx(fm, Pmilwd, '.', label='Sample')
plt.semilogx(ft, Gmilwv, label='Theoretical-MIL-STD')
plt.semilogx(ft, Swv, label='Theoretical-Dryden')
plt.vlines(fs/2, -80,20, label='50% sample frequency')
plt.legend()
plt.grid()
plt.xlabel('frequency (Hz)')
plt.ylabel('PSD (dB)')
plt.xlim([0.01, 100])
plt.ylim([-80, 20])

Compare to:


<img src='./img/f18_psd_w.png' width="40%" height="40%">